In [8]:
import pandas as pd
import numpy as np
from sklearn.utils import resample
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.backends.backend_pdf import PdfPages

# Load and preprocess data
def load_data(file_path):
    return pd.read_csv(file_path, sep='\t', encoding='utf-16')

# Define gene sets
gene_sets = {
    'Set1': ['Apitd1', 'CNIH4', 'exosc6', 'Gm10643', 'Gm9938', 'H2afx', 'INAFM1', 'rpl21', 'RPL32', 'RPLP2', 'TCEAL9'],
    'Set2': ['Sycp3', 'Meiob', 'Dnmt3b', 'Dnmt3c', 'Dnmt1', 'Dnmt3a', 'Ddx4', 'Dnmt3l', 'Nanos2', 'Nanos3'],
    'Set3': ['Piwil2', 'Piwil1', 'Piwil4']
}

# Function to annotate gene names on the plot with the color of their gene set
def annotate_genes_with_color(ax, df, gene_list, palette):
    for gene in gene_list:
        gene_data = df[df['Gene Name'].str.lower() == gene.lower()]
        for _, row in gene_data.iterrows():
            gene_color = palette[row['Gene Set']]
            ax.text(row['mean'], row['CV_squared'], row['Gene Name'], color=gene_color, fontsize=8, ha='right')

# Main analysis function
def perform_analysis(file_path, output_pdf_path):
    data = load_data(file_path)

    # Calculate statistics
    grouped_data = data.groupby(['Gene Name', 'timepoint'])['Norm_count'].agg(['mean', 'std']).reset_index()
    grouped_data['CV'] = grouped_data['std'] / grouped_data['mean']
    grouped_data['CV_squared'] = grouped_data['CV'] ** 2

    # Mark genes according to the sets
    grouped_data['Gene Set'] = grouped_data['Gene Name'].apply(lambda x: next((k for k, v in gene_sets.items() if x.lower() in [g.lower() for g in v]), 'Other'))

    # Determine hyper-variability threshold
    overall_hyper_thresholds = grouped_data.groupby('timepoint')['CV_squared'].quantile(0.95).reset_index(name='Overall Threshold')
    grouped_data = pd.merge(grouped_data, overall_hyper_thresholds, on='timepoint')
    grouped_data['is_overall_hyper_variable'] = grouped_data['CV_squared'] > grouped_data['Overall Threshold']

    # Set color palette for gene sets and plot results
    palette = {'Set1': 'red', 'Set2': 'blue', 'Set3': 'green', 'Other': 'gray'}
    with PdfPages(output_pdf_path) as pdf:
        for timepoint in grouped_data['timepoint'].unique():
            plt.figure(figsize=(12, 8))
            ax = plt.gca()
            tp_data = grouped_data[grouped_data['timepoint'] == timepoint]

            sns.scatterplot(data=tp_data, x='mean', y='CV_squared', hue='Gene Set', palette=palette, style='is_overall_hyper_variable', markers={True: 'X', False: 'o'}, alpha=0.6)
            sns.scatterplot(data=tp_data[tp_data['is_overall_hyper_variable']], x='mean', y='CV_squared', color='black', marker='X', label='Hyper-Variable', s=100)

            annotate_genes_with_color(ax, tp_data, sum(gene_sets.values(), []), palette)

            plt.title(f'Hyper-Variability Analysis at Timepoint {timepoint} with Colored Labels')
            plt.xlabel('Mean Normalized Count')
            plt.ylabel('CV Squared (CV²)')
            plt.xscale('log')
            plt.yscale('log')
            plt.legend(title='Gene Set / Hyper-Variable')
            plt.grid(True)
            pdf.savefig()
            plt.close()

# Specify your file path and output PDF path here
file_path = 'path_to_your_data/CV_varaintions_input.csv'
output_pdf_path = 'path_to_output/hyper_variability_plots.pdf'
perform_analysis(file_path, output_pdf_path)


In [3]:
pd.read_csv('/mnt/home3/miska/nm667/scratch/inProgress/mice_PiRNA/analysis/CV_analysis/CV_varaintions_input.csv', sep='\t')

,Timepoint,Norm_Count,Strains,Replicate,Gene_Name
0,E16.5,5290.978219,129S1_SvImJ,3,ZZZ3
1,E16.5,5258.317263,129S1_SvImJ,2,ZZZ3
2,E16.5,4792.120528,129S1_SvImJ,1,ZZZ3
3,E16.5,5418.021482,129S1_SvImJ,3,ZZEF1
4,E16.5,5091.425706,129S1_SvImJ,2,ZZEF1
...,...,...,...,...,...
2508501,P20.5,636.335746,WSB_EiJ,2,0610009B22Rik
2508502,P20.5,586.501406,WSB_EiJ,1,0610009B22Rik
2508503,P20.5,1186.038682,WSB_EiJ,3,0610007P14Rik
2508504,P20.5,1186.969709,WSB_EiJ,2,0610007P14Rik
